In [1]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('../src'))
from sklearn.model_selection import train_test_split
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from Transformer import Transformer
from PreProcessor import PreProcessor
from ModelCollection import ModelCollection
from PipelineBuilder import PipelineBuilder
from CrossValidation import CrossValidation

In [2]:
config_path = "../src/config.jsonc"
path_train, path_test = "../data/train.csv", "../data/test.csv"
data_loader = DataLoader(path_train, path_test)
train, test = data_loader.load()

In [3]:
test

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


In [4]:
train['SalePrice'] = np.log1p(train['SalePrice'])
data_splitter = DataSplitter("SalePrice", test_size=0.2)
X_train, X_test, y_train, y_test = data_splitter.split(train)
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0]/(X_train.shape[0]+X_test.shape[0])

((1168, 80), (292, 80), (1168,), (292,), 0.8)

In [5]:
# ML pipeline
transformer = Transformer(config_path=config_path)
preprocessor = PreProcessor(df=X_train, config_path=config_path)
preprocessor = preprocessor.build()
model_collection = ModelCollection()
pipelineBuilder = PipelineBuilder(transformer=transformer, preprocessor=preprocessor, model=None)

$\textbf{OLS}$

In [6]:
## OLS fit on X_train, y_train:

model = model_collection.get('OLS')
pipelineBuilder.model = model
pipeline = pipelineBuilder.build()
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()

           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.465753    4.498972
std      0.432727    0.539369    2.547491
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.075960
50%     11.945687   11.000000    4.951138
75%     12.250929   12.000000    6.778059
max     13.534474   13.000000   11.251881


In [7]:
## Cross-validation score

n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.13879510930280659), 'std_rmse': np.float64(0.031093300600202958)}


In [8]:
## Hyper-parameter tunning
param_grid = {
    "model__fit_intercept": [False, True]
}
search_ols = cv.hyper_param_tune(X, y, param_grid)

In [9]:
## Predicting on test set 
X_train, y_train = train.drop(columns=['SalePrice']), train['SalePrice']
pipeline.fit(X_train, y_train)
preds = np.expm1(pipeline.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_ols.csv", index=False)

$\textbf{Linear Regression (Regularized Models) - Ridge and Lasso}$

$Ridge:$

In [10]:
model = model_collection.get('Ridge')
pipelineBuilder.model = model
pipeline = pipelineBuilder.build()
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.469178    4.473828
std      0.432727    0.539577    2.511103
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.075960
50%     11.945687   11.000000    4.901415
75%     12.250929   12.000000    6.764743
max     13.534474   13.000000    9.108398


In [11]:
## Cross-validation score

n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.13392447817202835), 'std_rmse': np.float64(0.03017025654758295)}


In [12]:
## Hyper-parameter tunning
param_grid = {
    "model__alpha": [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.5, 2.0, 3, 5, 10, 20, 30, 40, 50, 100]
}
search_ridge = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_ridge.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results[['param_model__alpha', 'mean_test_score', 'std_test_score']]

,param_model__alpha,mean_test_score,std_test_score
6,1.5,-0.133863,0.029892
7,2.0,-0.133880,0.029720
5,1.1,-0.133899,0.030103
8,3.0,-0.133953,0.029545
4,0.9,-0.133962,0.030244
9,5.0,-0.134077,0.029480
3,0.7,-0.134093,0.030416
10,10.0,-0.134247,0.029672
2,0.5,-0.134365,0.030623
11,20.0,-0.134481,0.030018


In [25]:
coefs_values = search_ridge.best_estimator_.named_steps["model"].coef_
feature_names = search_ridge.best_estimator_.named_steps["preprocess"].get_feature_names_out()
coefs_df = pd.DataFrame({'Feature': feature_names, "Coefficient": coefs_values})
coefs_df = coefs_df.sort_values(by='Coefficient', key=abs, ascending=False)
coefs_df.head(20)

,Feature,Coefficient
59,cat__RoofMatl_ClyTile,-0.494190
82,cat__Condition2_PosN,-0.303637
36,cat__MSZoning_C (all),-0.277180
66,cat__RoofMatl_WdShngl,0.188376
173,cat__Neighborhood_StoneBr,0.124550
81,cat__Condition2_PosA,0.101245
60,cat__RoofMatl_CompShg,0.100522
28,num__GrLivArea,0.099119
167,cat__Neighborhood_NridgHt,0.097271
110,cat__Exterior1st_BrkComm,-0.097246


In [14]:
preds = np.expm1(search_ridge.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_ridge.csv", index=False)

$Lasso:$

In [15]:
model = model_collection.get('Lasso')
pipelineBuilder.model = model
pipeline = pipelineBuilder.build()
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

           y_test   pred       Diff%
count  292.000000  292.0  292.000000
mean    11.997654   12.0    2.780247
std      0.432727    0.0    2.309270
min     10.471978   12.0    0.007482
25%     11.751950   12.0    1.090029
50%     11.945687   12.0    2.110711
75%     12.250929   12.0    3.757012
max     13.534474   12.0   14.591530


In [16]:
## Cross-validation score

n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.3987817488470154), 'std_rmse': np.float64(0.025132411941850535)}


In [17]:
## Hyper-parameter tunning
param_grid = {
    "model__alpha": [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.5, 2.0, 3, 5, 10, 20, 30, 40, 50, 100]
}
search_lasso = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_lasso.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results[['param_model__alpha', 'mean_test_score', 'std_test_score']]

,param_model__alpha,mean_test_score,std_test_score
0,0.1,-0.215131,0.016083
1,0.3,-0.377950,0.029749
2,0.5,-0.398782,0.025132
3,0.7,-0.398782,0.025132
4,0.9,-0.398782,0.025132
5,1.1,-0.398782,0.025132
6,1.5,-0.398782,0.025132
7,2.0,-0.398782,0.025132
8,3.0,-0.398782,0.025132
9,5.0,-0.398782,0.025132


In [20]:
coefs_values = search_lasso.best_estimator_.named_steps["model"].coef_
feature_names = search_lasso.best_estimator_.named_steps["preprocess"].get_feature_names_out()
coefs_df = pd.DataFrame({'Feature': feature_names, "Coefficient": coefs_values})
coefs_df = coefs_df.sort_values(by='Coefficient', key=abs, ascending=False)
coefs_df.head(10)

,Feature,Coefficient
0,num__OverallQual,0.140461
28,num__GrLivArea,0.070859
34,num__GarageCars,0.032776
27,num__1stFlrSF,0.018211
206,ord_2__KitchenQual,0.014064
24,num__YearBuilt,0.005198
201,ord_1__BsmtQual,0.003019
1,num__BsmtHalfBath,0.000000
7,num__YrSold,-0.000000
6,num__MiscVal,0.000000


In [19]:
preds = np.expm1(search_lasso.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_lasso.csv", index=False)